In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path
)

documents = [file.parse() for file in reader.read()]
len(documents)

72

In [6]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

print(documents[0]['filename'])
print(documents[1]['filename'])
print(documents[2]['filename'])

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/02-environment.md
01-agentic-rag/lessons/03-rag.md


In [ ]:
import json
from evaluation_utils import llm_structured_retry


def generate_ground_truth(doc):

    user_prompt = json.dumps(doc)

    response, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in response.questions:
        results.append({
            "question": q,
            "filename": doc['filename']
        })

    return results, usage



In [ ]:
from evaluation_utils import calc_price, calc_total_price

usages = []
sum_inputs = 0.0
for i in range(3):
    results, usage = generate_ground_truth(documents[i])
    # print(f"{documents[i]['filename']} : {usage['input_tokens']}")
    print(f"{documents[i]['filename']} : {usage}")
    price = calc_price(usage)
    sum_inputs = sum_inputs + price["input_cost"]
    usages.append(usage)

# avg_inputs = usages.sum() / len(usages)

# avg_inputs

# print(sum_inputs)


01-agentic-rag/lessons/01-intro.md : ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=116, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1136)
01-agentic-rag/lessons/02-environment.md : ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=117, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1403)
01-agentic-rag/lessons/03-rag.md : ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=1280), output_tokens=99, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1852)
0.00304425


In [28]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [29]:
print(chunks[0])

{'start': 0, 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone

In [40]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [41]:
# Build a text Index
from minsearch import Index

tindex = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

tindex.fit(chunks)

In [31]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [42]:
texts = []

for doc in chunks:
    text = doc["content"]
    texts.append(text)

len(texts)

295

In [43]:
# Prepare the vectors
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/6 [00:00<?, ?it/s]

295

In [44]:
import numpy as np

X = np.array(vectors)

In [45]:
# Build a vector index
from minsearch import VectorSearch

vindex = VectorSearch(
    keyword_fields=["filename"]
)

vindex.fit(X, chunks)

In [75]:
def text_search(query, num_results=5):

    return tindex.search(
        query,
        num_results=num_results,
    )


In [76]:
def vector_search(query, num_results=5):
    qvector = model.encode(query)

    return vindex.search(
        qvector,
        num_results=num_results,
    )

In [79]:
q = ground_truth[0]["question"]

print(q)

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


In [86]:
text_results = text_search(q)

text_results

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [85]:
vector_results = vector_search(q)

vector_results

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [108]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))
    
    return relevance

In [109]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)

        relevance_total.append(relevance)

    return relevance_total

In [110]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt +1
    
    return cnt / len(relevance)

In [111]:
# Q4 Evaluating text search

relevance_text = compute_relevance_total(ground_truth, text_search)
hit_rate(relevance_text)


  0%|          | 0/360 [00:00<?, ?it/s]

0.7583333333333333

In [112]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [113]:
# Q5 Evaluating Vector search

relevance_vector = compute_relevance_total(ground_truth, vector_search)
mrr(relevance_vector)

  0%|          | 0/360 [00:00<?, ?it/s]

0.6356944444444446

In [127]:
def rrf(result_lists, k, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [128]:
def hybrid_search(query, k=200):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [116]:
# Q6 Evaluating Hybrid search and tuning

# print(hybrid_search(ground_truth[0]["question"])[0])
# ça retourne une liste avec un scoring fait entre text search et vector

relevance_vector = compute_relevance_total(ground_truth, hybrid_search)
# k = 60
mrr(relevance_vector)

  0%|          | 0/360 [00:00<?, ?it/s]

0.6721296296296295

In [120]:
# k = 1
relevance_vector = compute_relevance_total(ground_truth, hybrid_search)
mrr(relevance_vector)

  0%|          | 0/360 [00:00<?, ?it/s]

0.6722685185185188

In [123]:
# k = 50
relevance_vector = compute_relevance_total(ground_truth, hybrid_search)
mrr(relevance_vector)

  0%|          | 0/360 [00:00<?, ?it/s]

0.6721296296296295

In [126]:
# k = 100
relevance_vector = compute_relevance_total(ground_truth, hybrid_search)
mrr(relevance_vector)

  0%|          | 0/360 [00:00<?, ?it/s]

0.6721296296296295

In [129]:
# k = 200
relevance_vector = compute_relevance_total(ground_truth, hybrid_search)
mrr(relevance_vector)

  0%|          | 0/360 [00:00<?, ?it/s]

0.6721296296296295